# MLOps Overview

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 1/6

Training a model takes hours; keeping it useful takes a *system*. This lesson maps the whole MLOps landscape — why ML systems rot, what the lifecycle looks like, which tools fill which slot, and how mature your team really is.

## 🎯 Learning Objectives

- Define MLOps and explain why machine learning needs more than classic DevOps
- Diagnose **data drift** vs **concept drift** in a model that started failing
- Walk the six stages of the ML lifecycle and point at its feedback loop
- Match MLOps tool categories (tracking, registries, orchestration, serving, monitoring) to the job each does
- Locate a team on the Level 0–2 maturity ladder and pick the right next investment
- Extract monitoring lessons from a real-style incident postmortem

## 1. What Is MLOps?

MLOps (*Machine Learning Operations*) is the discipline of taking models out of notebooks and running them reliably in the real world — with versioning, automation, testing and monitoring around them.

Classic software mostly *reacts* to the world: tax rules change, you ship v2. An ML system is stranger — it is built **from** the world. The model's behaviour is baked in at training time from a snapshot of reality, and reality quietly moves on.

> **Analogy:** DevOps keeps a restaurant running. MLOps keeps a restaurant running when the customers' tastes change every season and the menu was printed a year ago.

**Syntax:** there is no API to memorise here — instead, internalise the equation every MLOps decision flows from:

```python
# A trained model is a function of BOTH code and data:
model = algorithm(training_data, hyperparameters)

# Change either ingredient and you have silently shipped a different model.
# MLOps = making all three parts (code, data, model) versioned, tested and watched.
```

In [ ]:
# Proof: the SAME code + DIFFERENT data = a DIFFERENT model
import numpy as np
from sklearn.linear_model import LinearRegression

hours = np.array([[1], [2], [3], [4], [5]])          # study hours

# Sarah revised in the morning vs at night — same person, different data
morning_scores = np.array([52, 58, 65, 70, 78])
night_scores = np.array([48, 61, 55, 74, 90])

m_morning = LinearRegression().fit(hours, morning_scores)
m_night = LinearRegression().fit(hours, night_scores)

new_hours = np.array([[3.5]])                        # always 2D: (n_samples, n_features)
p_m = m_morning.predict(new_hours)[0]
p_n = m_night.predict(new_hours)[0]
print(f"Predicted score after 3.5 h  (morning data): {p_m:.1f}")
print(f"Predicted score after 3.5 h  (night data):   {p_n:.1f}")

print("\nSame algorithm, same hyperparameters — only the DATA changed.")
print("So 'the model' is never just code. It is code + a frozen slice of the world.")

## 2. Why ML Systems Decay: Data Drift & Concept Drift

Software rots rarely and loudly (an exception). Models rot **constantly and silently**, because the statistical relationships they learned keep shifting. Two named kinds of shift matter:

| Drift type | What changed | Probability notation | Email-spam example |
|---|---|---|---|
| **Data drift** (covariate shift) | The *inputs* move; the input→label rule is unchanged | `P(X)` changes, `P(y \| X)` stays | After launch, legit marketing newsletters start containing "FREE" and many links — spammy-looking ham arrives |
| **Concept drift** | The *relationship* between inputs and labels moves | `P(y \| X)` changes | Spammers stop using ALL-CAPS and hide their links, so "looks shouty" stops meaning spam |

Both feel the same from outside: accuracy slides week after week while nothing crashes.

> 🔍 **Under the Hood:** At inference time your model is pure arithmetic — frozen weights multiplied by whatever features arrive. Nothing in the model knows the year, or that your company launched a new pricing plan. If the incoming feature distribution `P(X)` slides away from the training distribution, every prediction is still computed confidently against a boundary tuned for a world that no longer exists. There is no error to catch — the arithmetic is perfectly correct and perfectly obsolete.

In [ ]:
# A toy email world we can shift on purpose
import numpy as np
from sklearn.linear_model import LogisticRegression

FEATURES = ["links_per_100_words", "shouty_caps_ratio", "free_word_count"]


def make_emails(rng, n_spam, n_ham, spam_links, spam_caps, spam_free,
                ham_links, ham_caps, ham_free):
    """Synthesise emails as 3 features. Spammers push all three up; ham stays calm."""
    spam = np.column_stack([
        np.clip(rng.normal(spam_links, 3.0, n_spam), 0, None),
        np.clip(rng.normal(spam_caps, 0.10, n_spam), 0, 1),
        rng.poisson(spam_free, n_spam).astype(float),
    ])
    ham = np.column_stack([
        np.clip(rng.normal(ham_links, 3.0, n_ham), 0, None),
        np.clip(rng.normal(ham_caps, 0.04, n_ham), 0, 1),
        rng.poisson(ham_free, n_ham).astype(float),
    ])
    X = np.vstack([spam, ham])
    y = np.concatenate([np.ones(n_spam, dtype=int), np.zeros(n_ham, dtype=int)])
    return X, y


rng = np.random.default_rng(42)

# --- Month 1: the world the model learns on ---
X1, y1 = make_emails(rng, 600, 2400,
                     spam_links=13, spam_caps=0.45, spam_free=3.5,
                     ham_links=2, ham_caps=0.06, ham_free=0.1)
spam_clf = LogisticRegression(max_iter=2000).fit(X1, y1)

# Fresh sample from the SAME world = launch-time accuracy
X1b, y1b = make_emails(rng, 300, 1200,
                       spam_links=13, spam_caps=0.45, spam_free=3.5,
                       ham_links=2, ham_caps=0.06, ham_free=0.1)
print(f"Launch month accuracy (fresh Month-1 emails): {spam_clf.score(X1b, y1b):.2%}")

In [ ]:
# --- Month 6: the world moved ---
# Marketing discovered emojis-and-'FREE' newsletters; spammers started dressing like ham.
X6, y6 = make_emails(rng, 300, 1200,
                     spam_links=11.5, spam_caps=0.30, spam_free=2.4,  # partial camouflage
                     ham_links=7, ham_caps=0.09, ham_free=1.1)        # promos look spammy
print(f"Six months later, same model:  {spam_clf.score(X6, y6):.2%}   <- nobody restarted anything")

# Which features moved? Compare class means, Month 1 vs Month 6
import pandas as pd

def means_by_class(X, y):
    return pd.DataFrame(X, columns=FEATURES).assign(is_spam=y).groupby("is_spam").mean().round(2)

compare = pd.concat(
    {"month_1": means_by_class(X1, y1), "month_6": means_by_class(X6, y6)},
    names=["world"],
)
print(compare.swaplevel().sort_index())

print("""
Diagnosis:
- Ham's links/free words exploded ............ P(X) moved       -> DATA DRIFT
- Camouflaged spam broke the learned boundary . P(y|X) moved   -> CONCEPT DRIFT
Real decays are usually both at once. Neither throws an exception.""")

## 3. DevOps vs MLOps: What Actually Changes

If you already know DevOps, MLOps is *not* a foreign planet — it is DevOps carrying three extra bags: data, learned artefacts and statistical testing.

| Dimension | Classic software (DevOps) | ML system (MLOps) |
|---|---|---|
| Core artifact | Code | Code **+ data + learned weights** |
| Versioning | Git the code | Git code, **and** version data and models too (Lesson 02) |
| Testing | Unit/integration tests, exact assertions | Those **plus** data validation and metric gates (`recall >= 0.85`) — Lesson 05 |
| Monitoring | Errors, latency, CPU | Those **plus** input/output distributions and drift — Lesson 06 |
| Deploy trigger | Pull request merged | PR merged **or** metrics decayed **or** data shifted |
| Typical failure | Crashes loudly, page at 3 a.m. | Degrades silently for months |
| Team | Developers + ops | + data scientists, data engineers, domain experts |

The punchline row is the failure mode: broken software screams; broken models smile.

## 4. The ML Lifecycle: a Loop, Not a Line

Beginners imagine ML as *collect data → train → done*. Production ML is a circle you keep walking, and the most important arrow points backwards.

```text
   1. SCOPE          What problem? Would a rule work? Which metric matters?
      |
   2. DATA           Collect, label, validate, version  <-----------.
      |                                                              |
   3. MODEL          Train, track experiments, evaluate             |
      |                                                              |
   4. DEPLOY         Serve behind an API / batch job                |
      |                                                              |
   5. MONITOR        Inputs, outputs, latency, drift, KPIs          |
      |                                                              |
      +--- drift or decay? --yes--> 6. RETRAIN / REPAIR -------------+
                    |
                    no --> keep serving, check again tomorrow
```

| Stage | Question you answer | Key artifact | Skip it and… |
|---|---|---|---|
| 1. Scope | Is ML even the right tool? | Problem + metric definition | You automate a rule badly |
| 2. Data | Is the data good, labelled, understood? | Validated, versioned dataset | Garbage in, confident garbage out |
| 3. Model | Does it beat the baseline? | Tracked, evaluated model | You ship vibes |
| 4. Deploy | Can the product call it safely? | Versioned API / batch job | The model stays a demo forever |
| 5. Monitor | Is it still right *today*? | Dashboards + alerts | You learn about decay from angry users |
| 6. Retrain | Fix features, labels, weights? | New validated model | The decay compounds |

In [ ]:
# Walk the loop — notice stage 6 feeds straight back into stage 2
LIFECYCLE = [
    ("1. scope", "problem + success metric"),
    ("2. data", "validated, versioned dataset"),
    ("3. model", "tracked + evaluated model"),
    ("4. deploy", "versioned API endpoint"),
    ("5. monitor", "dashboards, alerts, drift reports"),
    ("6. retrain", "candidate model -> loops back to DATA"),
]

for stage, artifact in LIFECYCLE:
    print(f"{stage:<12} produces: {artifact}")

print("\nThe loop never ends: every 'done' is only valid until the world moves.")

## 5. The Component Landscape

"MLOps" is an umbrella over several distinct tools. Learn the *slot* each fills before caring about brand names — then the tool choice becomes easy.

| Component | The job it does | Example tools | Covered in |
|---|---|---|---|
| Experiment tracking | Record every training run's params, metrics, artefacts | MLflow, Weights & Biases, Neptune | Lesson 03 |
| Data versioning | Snapshot datasets like git commits | DVC, LakeFS, Delta Lake | Lesson 02 |
| Model registry | Version + promote models (staging → production) | MLflow Registry, SageMaker Registry | Lesson 03 |
| Orchestration | Run multi-step pipelines on schedule | Airflow, Prefect, Dagster, Kubeflow | Lesson 06 (concepts) |
| Serving | Expose the model to applications | FastAPI, BentoML, TF Serving, KServe | Lesson 04 |
| Monitoring | Watch data quality, drift and business impact | Evidently, Grafana, Arize, WhyLabs | Lesson 06 |

> **Day-one stack:** git + a requirements file + one experiment tracker + one drift chart. Every other box waits until its pain is real.

## 6. Maturity Levels 0–2: Where Is Your Team?

Google popularised three maturity levels. The levels are not about budget — they are about **how many human hands a model touches on its way to production**.

| Level | Name | What production looks like |
|---|---|---|
| 0 | Manual | A notebook *is* the pipeline; someone runs it, exports a CSV, emails the model file |
| 1 | Pipeline automation | Training is one automated, repeatable pipeline; registry + basic monitoring exist; retraining is still triggered by humans |
| 2 | CI/CD + CT | Tests, data validation and evaluation gates run automatically; deploying *and* retraining happen without hand-holding (CT = continuous training) |

**Symptoms you are at Level 0:** "Which notebook produced the live model?" gets a shrug · deploys happen by copying files · metrics live in screenshots · nobody notices decay for months.

**Symptoms you are at Level 1:** retraining is one scripted command · experiments are tracked · drift dashboard exists · but promotion to production is a meeting, not a merge.

**Symptoms you are at Level 2:** a pull request runs the whole test suite · a failing metric gate blocks the release · retraining fires on drift · rollback is one click.

In [ ]:
# Self-check: place a team on the ladder
def maturity_level(has_pipeline=False, has_tracking=False, has_monitoring=False,
                   has_registry=False, has_ci=False, auto_retrain=False):
    if auto_retrain and has_ci and has_monitoring:
        return "Level 2 — CI/CD + continuous training"
    if has_pipeline and (has_tracking or has_monitoring):
        return "Level 1 — automated pipeline, human-triggered retraining"
    return "Level 0 — manual notebook territory"


# Simulated input: three teams answer the same checklist
teams = {
    "DhakaTel (before this module)": dict(),
    "DhakaTel (after Lesson 03)": dict(has_pipeline=True, has_tracking=True, has_monitoring=True),
    "Payments unicorn": dict(has_pipeline=True, has_tracking=True, has_monitoring=True,
                             has_registry=True, has_ci=True, auto_retrain=True),
}

for team, answers in teams.items():
    print(f"{team:<32} -> {maturity_level(**answers)}")

print("\nThe goal is NOT Level 2 on day one. It is knowing your level and fixing the loudest pain.")

## 7. Who Does What: Roles on an ML Team

| Role | Owns | Typical question |
|---|---|---|
| Data scientist | Modelling, features, evaluation design | "Will gradient boosting beat this ruleset?" |
| ML engineer | Training + serving code, packaging | "How does this model become a reliable API?" |
| Data engineer | Pipelines, warehouses, data quality | "Is tonight's ingestion complete and correct?" |
| MLOps / platform engineer | Infrastructure, CI/CD, monitoring | "Why did the nightly training fail, and how do we roll back?" |
| Product manager | Problem choice, metric ownership | "Did the model move the business number?" |
| Domain expert | Labels, edge cases, sanity | "Would a doctor agree with this prediction?" |

At a five-person startup these are hats, not people — but every hat must be worn *by someone*, especially monitoring after launch.

## 8. Mini Case Study: The Churn Model That Died Quietly

> **Postmortem — DhakaTel prepaid-churn model**
>
> **What we shipped:** a gradient-boosting churn model (March). Offline recall **0.88** on a February holdout. Flagged subscribers get a retention offer by SMS.
>
> **Timeline**

| Month | Event | Dashboard said |
|---|---|---|
| Mar | Model deployed, recall 0.88 at launch | All green |
| Apr–Aug | Offer uptake slowly fell; nobody connected it to the model | All green (no new labels yet) |
| Sep | Quarterly labels finally land: **recall is 0.41** | Red, five months late |
| Oct | Root cause found (below) | — |

> **Root cause:** a competitor launched an unlimited-data plan. Customers began churning for *price* reasons the model had never seen, while a new IVR flow silently cut logged support calls — the model's strongest feature — roughly in half.
>
> **Contributing factors:** no input monitoring (the support-call collapse was visible in June) · labels arrive quarterly, so recall was blind for months · retraining required a human with a notebook.
>
> **Action items:** track every run (Lesson 03) · monitor input distributions + alert on shift (Lesson 06) · add proxy metrics that update daily despite slow labels · automate the retrain pipeline (Lessons 02 & 06).

In [ ]:
# Rebuild the fatal blind spot: what the dashboards showed vs reality
import numpy as np
import pandas as pd

postmortem = pd.DataFrame({
    "month": ["Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"],
    "share_flagged_as_churners": [0.210, 0.196, 0.181, 0.170, 0.158, 0.149, 0.141],
    "fresh_labels_available": ["yes", "no", "no", "no", "no", "no", "yes"],
    "measured_recall": [0.88, np.nan, np.nan, np.nan, np.nan, np.nan, 0.41],
})
postmortem["measured_recall"] = postmortem["measured_recall"].map(
    lambda v: "--" if np.isnan(v) else f"{v:.2f}"
)
print(postmortem.to_string(index=False))

drop = 1 - 0.141 / 0.210
print(
    f"\nThe ONLY visible clue: flagged-share slid {drop:.0%} between Mar and Aug,"
    "\nwhile every 'green' dashboard waited for labels that came quarterly."
    "\nInput-distribution monitoring would have screamed in June."
)

## 9. Where This Module Goes

| Lesson | Folder | You leave able to… |
|---|---|---|
| 2 | [02_Reproducibility_Environments](../02_Reproducibility_Environments/notes.ipynb) | Make any training run replayable: seeds, pinned deps, config files, versioned data |
| 3 | [03_Experiment_Tracking_MLflow](../03_Experiment_Tracking_MLflow/notes.ipynb) | Track experiments (with MLflow *and* an offline file-based tracker) and pick winners by query |
| 4 | [04_Model_Serving_FastAPI](../04_Model_Serving_FastAPI/notes.ipynb) | Wrap a model in a validated HTTP API and containerise it |
| 5 | [05_Testing_Data_Quality](../05_Testing_Data_Quality/notes.ipynb) | Test code, data *and* model behaviour; block bad data and bad models |
| 6 | [06_CI_CD_And_Monitoring](../06_CI_CD_And_Monitoring/notes.ipynb) | Automate the path to production and catch drift with PSI before users do |

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| "Monitoring comes after launch" | Drift starts with the very first real user; you are blind on day one | Instrument input logging *before* the first request goes out |
| Judging health by offline accuracy alone | The holdout is a snapshot; it agrees with you right up until it doesn't | Monitor live input/output distributions, not just periodic benchmarks |
| Assuming scheduled retraining fixes drift | If the *feature* broke (IVR change), retraining bakes the bug in | Triage drift first: fix data/features, then consider retraining |
| Copying a big-tech stack at day one | Six platforms, zero models shipped | Buy tools matching your maturity level; upgrade when pain, not fashion, says so |
| Treating handover to ops as "done" | Ownership vacuum: scientists think ops watches, ops thinks science watches | Named model owner + on-call rota in the runbook |
| Celebrating the launch metric forever | Launch-week data is the easiest data you will ever see | Re-baseline quarterly against fresh labels |

## 💡 Best Practices & Pro Tips

- **Automate the painful thing you did twice this month** — not the impressive thing you saw in a conference talk.
- **Log inputs, not just outputs.** Prediction logs double as tomorrow's training data and today's drift evidence.
- **Define "the model is wrong" operationally**: which metric, which threshold, who gets paged. Vague SLOs decay into no SLOs.
- **Keep a dumb baseline alive** (majority class, one rule). When your fancy model barely beats it in monitoring, that is information.
- **AI-engineering relevance:** LLM apps inherit every rule here — prompts are configs to version, generations are outputs to monitor, and "the vendor updated the model" is concept drift arriving without a deploy.
- **Write the postmortem while you ship.** Listing "how could this fail silently?" at launch time is cheaper than a September surprise.

## 📌 Summary

| Term / Concept | What it means | Example |
|---|---|---|
| MLOps | Engineering discipline for the full ML lifecycle | Seeds + tracking + serving + monitoring around one model |
| Data drift | Input distribution shifts, `P(X)` | Legit newsletters become link-heavy post-launch |
| Concept drift | Input→label relationship shifts, `P(y\|X)` | Spammers stop using ALL-CAPS |
| Lifecycle loop | scope→data→model→deploy→monitor→retrain | Stage 6 feeds stage 2, forever |
| Maturity L0/L1/L2 | manual → pipeline automation → CI/CD+CT | Copy-pasting files vs one-click rollback |
| Model registry | Versioned home with staging/production stages | Promote v3 after the gate passes |
| Canary / shadow | Release patterns that limit blast radius | 5% of traffic first (Lesson 06) |

Key takeaways:

- A model is code + a frozen snapshot of the world; the unfreezing is called drift.
- Broken software crashes; broken models degrade politely — monitoring is not optional.
- Tools fill slots (track, version, orchestrate, serve, watch); choose by pain, not by logo.
- Know your maturity level; automate the loudest pain next.

## 🔗 Next Lesson

Up next: **[02_Reproducibility_Environments](../02_Reproducibility_Environments/notes.ipynb)** — the unglamorous superpower: making any run replayable bit-for-bit with seeds, pins, configs and data snapshots.